# Reinforcement Learning

# 7. Parametric Bandits


The objective of this lab is to recommend contents (here movies) using **parametric bandits**. The rewards are binary (like or dislike).


## Imports


In [1]:
import numpy as np
import pandas as pd

You will need `ipywidgets` to simulate the interactions with the user.


In [2]:
#!pip install ipywidgets

In [3]:
from ipywidgets import AppLayout, Button, GridspecLayout, Image, Layout

In [4]:
#!pip install scikit-learn

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer

## Data


We work on a catalogue of 1037 movies available in 2015.


In [6]:
catalogue = pd.read_pickle("movie_database.pickle")

In [ ]:
len(catalogue)

In [ ]:
catalogue.head()

The features are the following:

| Column     | Description                                      | Type             |
| :--------- | :----------------------------------------------- | :--------------- |
| Actors     | Actors staring                                   |  list of strings |
| Awards     | Awards received                                  |  string          |
| Country    | Country of origin                                | list of strings  |
| Director   | Director(s) of the movie                         | list of strings  |
| Genre      | Genres (Action, ...)                             | list of strings  |
| Language   | Language(s) spoken                               | list of strings  |
| Rated      | Public rating (G = General, R = Restricted, ...) |  list of strings |
| Released   | Date of the movie                                |  date            |
| Title      | Title of the movie                               | string           |
| imdbID     | IMDB id                                          | string           |
| imdbRating | IMDB rating (between 0 and 10)                   |  float           |
| Metascore  | Metacritic score (between 0 and 100)             | float            |
| Box_office | Total money generated                            | float            |
| imdbVotes  | Number of IMDB votes                             |  float           |
| Runtime    |  Duration of the movie (in minutes)              | float            |
| poster     |  Poster of the movie (jpg)                       |  binary string   |


In [9]:
# Display the posters

def get_poster(k, scale=1):
    return Image(
        value=catalogue.loc[k].poster,
        format="jpg",
        width=130 * scale,
        height=200 * scale,
    )

def display_posters(index=None, n_col=5, n_rows=4):
    if index is None:
        index = np.arange(len(catalogue))
    if len(index):
        n_rows = min(n_rows, int(np.ceil(len(index) / n_col)))
        grid = GridspecLayout(n_rows, n_col)
        k = 0
        for i in range(n_rows):
            for j in range(n_col):
                if k < len(index):
                    grid[i, j] = get_poster(index[k])
                k += 1
        return grid

In [ ]:
display_posters()

## Features

We will describe each movie by some features, for instance its genre.


In [ ]:
mlb = MultiLabelBinarizer()
movies = pd.DataFrame(mlb.fit_transform(catalogue["Genre"]), columns=mlb.classes_)
movies.head()

In [ ]:
movies.columns

## User

Each user will be modeled by a vector of weights (positive or negative) on each feature.


In [13]:
user = pd.DataFrame(0, index=[0], columns=movies.columns)
user["Action"] = 2
user["Crime"] = 1
user["Sci-Fi"] = -2

## To do

- Display the favorite movies of this user.
- Test another user, and quantify their similarity (e.g., proportion of common top-100 movies).


### Favorite movies of user

In [ ]:
scores_user1 = np.array(movies).dot(np.array(user).flatten())
favorite_movies_user1 = np.argsort(-scores_user1)
display_posters(favorite_movies_user1)

### Another user

In [15]:
user2 = pd.DataFrame(0, index=[0], columns=movies.columns)
user2["Thriller"] = 2
user2["Sci-Fi"] = 1
user2["Musical"] = -2

In [ ]:
scores_user2 = np.array(movies).dot(np.array(user2).flatten())
favorite_movies_user2 = np.argsort(-scores_user2)
display_posters(favorite_movies_user2)

### Similarity between users

In [ ]:
common_movies = np.intersect1d(favorite_movies_user1[:100], favorite_movies_user2[:100])
similarity = len(common_movies) / 100
print(f"Similarity between users: {similarity*100:.2f}%")

## Offline learning

We start with offline learning. There are 2 steps:

1. Collect the user's opinion on a few movies (e.g., 10)
2. Rank the other movies by logistic regression.

Let's test that.


In [18]:
# Add a column to record the user's opinion (like / dislike)
movies = movies.assign(like=None)

In [19]:
# Select a random movie (not yet seen by the user)

def select_random_movie(movies):
    index = np.flatnonzero(movies.like.isna())
    if len(index):
        return np.random.choice(index)
    else:
        return np.random.choice(len(movies))

In [20]:
# Create buttons

def create_expanded_button(description, button_style):
    return Button(description=description, button_style=button_style, layout=Layout())


def update_likes(button):
    global movie_id
    movies.loc[movie_id, "like"] = button.description == "like"


def update_poster():
    global movie_id
    img.value = catalogue.loc[movie_id].poster


def on_button_clicked(button):
    global movie_id
    update_likes(button)
    movie_id = select_random_movie()
    update_poster()

In [ ]:
# Setting the buttons
left_button = create_expanded_button("like", "success")
right_button = create_expanded_button("dislike", "danger")
left_button.on_click(on_button_clicked)
right_button.on_click(on_button_clicked)

# Setting the movie poster
movie_id = select_random_movie(movies)
img = get_poster(movie_id, scale=1.5)

# Display
AppLayout(
    left_sidebar=left_button,
    right_sidebar=right_button,
    center=img,
    pane_widths=[0.3, 0.4, 0.3],
)

## To do

- Give your opinion on some movies (e.g., 10), making sure that you get a few likes and a few dislikes.
- Apply logistic regression and display the other movies in order of preference (top movies first).
- Give your top-3 and bottom-3 genres, as predicted by the model.

### Opinion on movies

In [22]:
# likes
likes = [   6,   19,   20,   34,  224,  375,  415,  417,  461,  502,  507,
        552,  568,  570,  611,  667,  785,  805,  825,  842,  877,  885,
        908,  927,  987, 1009, 1028]
for index in likes:
        movies.at[index, 'like'] = True

# dislikes
dislikes = [  56,   76,  130,  193,  199,  211,  213,  320,  335,  394,  427,
        429,  534,  626,  628,  640,  662,  678,  688,  695,  702,  723,
        731,  739,  767,  770,  772,  807,  809,  911,  913,  953,  960,
        970, 1021, 1029]
for index in dislikes:
        movies.at[index, 'like'] = False

### Logistic regression

In [23]:
# Separating the movies with labeled preferences and those without
labeled_movies = movies.copy()[movies['like'].notna()]
unlabeled_movies = movies.copy()[movies['like'].isna()]
unlabeled_movies.drop(columns="like", inplace=True)

# Apply logistic regression
model = LogisticRegression(fit_intercept=False)
model.fit(labeled_movies.drop(columns="like"), labeled_movies["like"].astype(int))

# Predict the probability of liking the movies
unlabeled_movies.loc[:, "preference_score"] = model.predict_proba(unlabeled_movies)[:, 1]

# Sort the movies by preference score
recommended_movies = unlabeled_movies.sort_values("preference_score", ascending=False)

In [ ]:
# Display movies
print("Top 20 movies indexes:", list(recommended_movies.index[:20]))
display_posters(recommended_movies.index[:20])

### Top 3 and bottom 3 genres

In [ ]:
top3_genres = (-model.coef_[0]).argsort()[:3]
bottom3_genres = model.coef_[0].argsort()[:3]
genres = movies.drop(columns="like").columns
total_genres = len(genres)

print("Top 3 genres:\n1.", genres[top3_genres[0]], "\n2.", genres[top3_genres[1]], "\n3.", genres[top3_genres[2]])
print(f"\nBottom 3 genres:\n{total_genres-2}.", genres[bottom3_genres[2]], f"\n{total_genres-1}.", genres[bottom3_genres[1]], f"\n{total_genres}.", genres[bottom3_genres[0]])

## Online learning


We now learn the user preferences online, as they come. For that, we use a Bayesian algorithm inspired by Thompson sampling.

On each feedback provided by the user:

1. (Learning) The parameter (vector of weights) is learned.
2. (Sampling) A new parameter is sampled, assuming a Gaussian distribution.
3. (Action) The top movie for this new parameter, among movies not yet seen by the user, is proposed.

Note that:

- In step 1, we retrain the estimator **from scratch**, using logistic regression on all training data samples (**no** online estimation).
- In step 2, we discard correlations (**diagonal** covariance matrix).


## To do

- Complete the function `select_bayes` below.
- Test it on some movies (e.g., 10), until you get a few likes and a few dislikes.
- Display the other movies in order of preference (top movies first).


In [26]:
model = LogisticRegression(fit_intercept=False)

In [27]:
def select_bayes(movies):

    if set(movies.like) == {True, False, None}:
        # to be completed (learning, sampling, action)
        labeled_movies = movies.copy()[movies['like'].notna()]
        unlabeled_movies = movies.copy()[movies['like'].isna()]
        unlabeled_movies.drop(columns="like", inplace=True)

        # Learning
        X = np.array(labeled_movies.drop(columns="like"))
        y = np.array(labeled_movies["like"].astype(int))

        model.fit(X, y)

        # Sampling
        mu = model.coef_.flatten()
        probs = model.predict_proba(X)[:, 1]
        sigma = np.diag(probs * (1 - probs))
        H = (X.T @ sigma) @ X + np.eye(X.shape[1])
        cov = np.linalg.inv(H)
        cov = np.diag(np.diag(cov))

        new_mu = np.random.multivariate_normal(mu, cov)

        # Action
        scores = unlabeled_movies @ new_mu
        recommended_movie = scores.idxmax()
        
        return recommended_movie

    else:
        return select_random_movie(movies)

In [28]:
# reset
movies = movies.assign(like=None)

In [29]:
def on_button_clicked(button):
    global movie_id
    update_likes(button)
    movie_id = select_bayes(movies)
    update_poster()

In [ ]:
# Setting the buttons
left_button = create_expanded_button("like", "success")
right_button = create_expanded_button("dislike", "danger")
left_button.on_click(on_button_clicked)
right_button.on_click(on_button_clicked)

# Setting the movie poster
movie_id = select_random_movie(movies)
img = get_poster(movie_id, scale=1.5)

# Display
AppLayout(
    left_sidebar=left_button,
    right_sidebar=right_button,
    center=img,
    pane_widths=[0.3, 0.4, 0.3],
)

In [31]:
# likes
likes = [5, 14, 26, 34, 72, 182, 216, 324, 337, 449, 921]
for index in likes:
        movies.at[index, 'like'] = True

# dislikes
dislikes = [  13,  120,  279,  332,  347,  357,  412,  421,  422,  519,  524,  571,
        582,  596,  597,  731,  753,  859,  931,  979, 1007, 1030]
for index in dislikes:
        movies.at[index, 'like'] = False

In [ ]:
# Separating the movies with labeled preferences and those without
labeled_movies = movies.copy()[movies['like'].notna()]
unlabeled_movies = movies.copy()[movies['like'].isna()]
unlabeled_movies.drop(columns="like", inplace=True)

# Apply logistic regression
model = LogisticRegression(fit_intercept=False)
model.fit(labeled_movies.drop(columns="like"), labeled_movies["like"].astype(int))

# Predict the probability of liking the movies
unlabeled_movies.loc[:, "preference_score"] = model.predict_proba(unlabeled_movies)[:, 1]

# Sort the movies by preference score
recommended_movies = unlabeled_movies.sort_values("preference_score", ascending=False)

# Display movies
print("Top 20 movies indexes:", list(recommended_movies.index[:20]))
display_posters(recommended_movies.index[:20])

## Analysis

Finally, we would like to assess the quality of our bandit algorithm.

## To do

- Choose a user, that is a parameter $\theta$ (vector of weights).
- Provide the answers of this user to the movies proposed by the algorithm, assuming binary rewards, with mean
  $$
  q(a) = \frac 1 {1 + e^{-\theta^T a}}
  $$
  where $a$ is the action (= movie proposed by the algorithm).
- Make sure that a reasonable fraction of movies are liked (e.g., between 10\% and 90\%). Otherwise, update $\theta$.
- Simulate an interaction of this user with the recommender system over 100 movies.
- Compute the [Spearman's correlation coefficient](https://en.wikipedia.org/wiki/Spearman%27s_rank_correlation_coefficient) of the ranking of the unseen movies provided by the algorithm, compared to the ground-truth ranking.
- Plot the evolution of this coefficient with respect to the number of movies seen by the user, from 1 to 100.
- Give the top-3 and bottom-3 genres, as predicted by the model, and compare to the ground-truth.
- Do the same experiments with other features (e.g., actors, actors + genres, actors + director + genres).


### Setup

In [ ]:
model = LogisticRegression(fit_intercept=False)

# Reset
movies = movies.assign(like=None)

# Select random user
user = np.random.randn(movies.drop(columns="like").shape[1])

# Check for 10% to 90% liked movies
q_a = 1 / (1 + np.exp(-movies.drop(columns="like").dot(user)))
while 0.1 > np.mean(q_a) > 0.9:
    user = np.random.randn(movies.drop(columns="like").shape[1])
    q_a = 1 / (1 + np.exp(-movies.drop(columns="like").dot(user)))

print("User likes", round(np.mean(q_a) * 100, 2), "% of the movies")

### Simulation

In [34]:
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

def simulation(movies, n_movies=100):
    spearman_corrs = []

    for i in range(n_movies):
        movie_id = select_bayes(movies)

        # User's opinion
        movie = movies.loc[movie_id].drop("like")
        q_a = 1 / (1 + np.exp(-movie.dot(user)))
        opinion = np.random.binomial(1, q_a)
        movies.loc[movie_id, "like"] = opinion

        # Rank unlabeled movies
        if not hasattr(model, "coef_"):
            continue
    
        unlabeled_movies = movies.copy()[movies['like'].isna()]
        predictions = model.predict_proba(np.array(unlabeled_movies.drop(columns="like")))[:, 1]
        gt = 1 / (1 + np.exp(-unlabeled_movies.drop(columns="like").dot(user)))

        spearman_corr, _ = spearmanr(gt, predictions)
        spearman_corrs.append(spearman_corr)

    return spearman_corrs

def plot_simulation(spearman_corrs, title="Spearman correlation between true and predicted preferences"):
    plt.figure(figsize=(10, 5))
    plt.plot(spearman_corrs)
    plt.xlabel("Number of movies rated")
    plt.ylabel("Spearman correlation")
    plt.title(title)
    plt.show()

### Spearman correlation evolution

In [ ]:
spearman_correlations = simulation(movies)
plot_simulation(spearman_correlations)

In [ ]:
genres = movies.drop(columns="like").columns
total_genres = len(genres)

top3_model = (-model.coef_[0]).argsort()[:3]
bottom3_model = model.coef_[0].argsort()[:3]

top3_ground_truth = (-user).argsort()[:3]
bottom3_ground_truth = user.argsort()[:3]

print("Model Top 3 genres: 1.", genres[top3_model[0]], "2.", genres[top3_model[1]], "3.", genres[top3_model[2]])
print("Ground Truth Top 3 genres: 1.", genres[top3_ground_truth[0]], "2.", genres[top3_ground_truth[1]], "3.", genres[top3_ground_truth[2]])
print(f"\nModel Bottom 3 genres: {total_genres-2}.", genres[bottom3_model[2]], f"{total_genres-1}.", genres[bottom3_model[1]], f"{total_genres}.", genres[bottom3_model[0]])
print(f"Ground Truth Bottom 3 genres: {total_genres-2}.", genres[bottom3_ground_truth[2]], f"{total_genres-1}.", genres[bottom3_ground_truth[1]], f"{total_genres}.", genres[bottom3_ground_truth[0]])

## Other features

### Actors

In [ ]:
model = LogisticRegression(fit_intercept=False)

actor_movies = pd.DataFrame(mlb.fit_transform(catalogue["Actors"]), columns=mlb.classes_)
actor_movies = actor_movies.assign(like=None)

# Select random user
user = np.random.randn(actor_movies.drop(columns="like").shape[1])

# Check for 10% to 90% liked movies
q_a = 1 / (1 + np.exp(-actor_movies.drop(columns="like").dot(user)))
while 0.1 > np.mean(q_a) > 0.9:
    user = np.random.randn(actor_movies.drop(columns="like").shape[1])
    q_a = 1 / (1 + np.exp(-actor_movies.drop(columns="like").dot(user)))

print("User likes", round(np.mean(q_a) * 100, 2), "% of the movies")

In [ ]:
spearman_correlations = simulation(actor_movies, n_movies=10)
plot_simulation(spearman_correlations)